# Request Bodies and Responses

This notebook covers:

1. Accept JSON request bodies typed by Pydantic models
2. Return Pydantic models and let FastAPI handle serialization
3. Mix path/query params with a body in the same endpoint
4. Customize `response_model_exclude_unset` and friends
5. Drop to a raw `Response` when JSON is not the answer

**Scope**: FastAPI + Pydantic v2. Each section spins up its own tiny app.

## 1. JSON In, JSON Out

By default a FastAPI endpoint speaks JSON in and out. The body of an incoming request is parsed from JSON into Pydantic models; the return value of your handler is serialized to JSON.

That default is right ~95% of the time. The other 5% — CSV exports, streaming, redirects, file downloads — needs a different content type, and we'll see how to hand-roll those in §6.

## 2. Body Models

A handler argument typed as a `BaseModel` is interpreted as the request body. Everything that isn't path or body becomes a query param.

We'll use two models for the asset resource:
- **`AssetIn`** — what clients send. No internal fields.
- **`AssetOut`** — what we return. Same shape minus anything sensitive.

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

class AssetIn(BaseModel):
    ticker: str = Field(min_length=1, max_length=10, pattern=r"^[A-Z.]+$")
    name: str = Field(min_length=1)
    price: float = Field(ge=0)

class AssetOut(BaseModel):
    ticker: str
    name: str
    price: float

app = FastAPI()

@app.post("/assets", response_model=AssetOut, status_code=201)
def create_asset(asset: AssetIn):
    # Pretend storage adds internal bookkeeping fields.
    record = asset.model_dump() | {"internal_score": 0.9, "audit_id": "x-42"}
    return record  # internal fields stripped by response_model

client = TestClient(app)

r = client.post("/assets", json={"ticker": "AAPL", "name": "Apple Inc.", "price": 195.0})
print("status:", r.status_code, "body:", r.json())

## 3. Combining Body + Path + Query

In one handler you can have all three. FastAPI disambiguates by type:

- argument named in the path → **path param**
- argument with a `BaseModel` type → **body**
- everything else → **query param**

A common shape: `PUT /assets/{ticker}` with the new representation in the body, plus a `?dry_run=true` query flag for staged updates.

In [ ]:
app = FastAPI()

@app.put("/assets/{ticker}", response_model=AssetOut)
def replace_asset(ticker: str, asset: AssetIn, dry_run: bool = False):
    # If dry_run, don't mutate state — but the response model still applies.
    if dry_run:
        return asset.model_dump() | {"ticker": ticker.upper()}
    # In a real app this would persist the new representation.
    return asset.model_dump() | {"ticker": ticker.upper()}

client = TestClient(app)

r = client.put("/assets/aapl?dry_run=true", json={"ticker": "AAPL", "name": "Apple Inc.", "price": 195.0})
print("PUT /assets/aapl?dry_run=true ->", r.status_code, r.json())

## 4. Multiple Body Parameters

You can ask for two Pydantic models in the same handler — FastAPI wraps them in a single JSON object keyed by the parameter names:

```json
{
  "asset": { ... AssetIn ... },
  "audit": { ... AuditMeta ... }
}
```

For scalar singletons in the body (`reason: str`), wrap with `Body(...)` so FastAPI doesn't treat them as query params. Use this sparingly — usually one model with a nested field is clearer than two models.

In [ ]:
from fastapi import Body
from typing import Annotated

class AuditMeta(BaseModel):
    actor: str
    source: str

app = FastAPI()

@app.post("/assets")
def create_asset(
    asset: AssetIn,
    audit: AuditMeta,
    reason: Annotated[str, Body(min_length=1)],
):
    return {
        "asset": asset.model_dump(),
        "audit": audit.model_dump(),
        "reason": reason,
    }

client = TestClient(app)

body = {
    "asset":  {"ticker": "AAPL", "name": "Apple Inc.", "price": 195.0},
    "audit":  {"actor": "trader-1", "source": "manual"},
    "reason": "initial onboarding",
}
r = client.post("/assets", json=body)
print("status:", r.status_code)
print("body  :", r.json())

## 5. `response_model` Filtering and Aliases

Beyond stripping fields, `response_model` has knobs that earn their keep for PATCH-style endpoints and field renaming.

- **`response_model_exclude_unset=True`** — omit any field the handler didn't explicitly set. This is what makes PATCH responses correctly omit untouched fields instead of returning defaults.
- **`response_model_exclude={...}`** / **`response_model_include={...}`** — per-route field filtering.
- **`Field(alias=...)` + `populate_by_name=True`** — accept and emit a JSON name that differs from the Python attribute name (e.g., emit `displayName` while the field is `display_name`).

In [ ]:
from pydantic import ConfigDict

class AssetPatch(BaseModel):
    # Every field is optional — PATCH means "change what's set, leave the rest."
    name: str | None = None
    price: float | None = None

class AssetView(BaseModel):
    model_config = ConfigDict(populate_by_name=True)

    ticker: str
    display_name: str = Field(alias="displayName")
    price: float

app = FastAPI()

STORE = {"AAPL": {"ticker": "AAPL", "display_name": "Apple Inc.", "price": 195.0}}

@app.patch("/assets/{ticker}", response_model=AssetPatch, response_model_exclude_unset=True)
def patch_asset(ticker: str, patch: AssetPatch):
    updates = patch.model_dump(exclude_unset=True)
    STORE[ticker].update(updates)
    return updates  # only the fields that were actually patched

@app.get("/assets/{ticker}", response_model=AssetView, response_model_by_alias=True)
def get_asset(ticker: str):
    return STORE[ticker]

client = TestClient(app)

# PATCH only sends `price` — response includes only `price`.
r = client.patch("/assets/AAPL", json={"price": 200.0})
print("PATCH /assets/AAPL price=200    ->", r.json())

# GET emits the alias `displayName`, not the Python attribute `display_name`.
r = client.get("/assets/AAPL")
print("GET   /assets/AAPL (aliased)    ->", r.json())

## 6. Returning `Response` Directly (when you need to)

JSON isn't always the right answer. CSV exports, file downloads, custom content types, no-content responses — they all want raw control. FastAPI gives you `Response` (and subclasses `PlainTextResponse`, `HTMLResponse`, `StreamingResponse`) to bypass `response_model` and set everything explicitly.

Returning a `Response` from a handler:
- skips `response_model` shaping
- skips automatic JSON encoding
- lets you set `media_type`, `status_code`, and `headers`

In [ ]:
from fastapi import Response

app = FastAPI()

PORTFOLIO = [
    {"ticker": "AAPL", "shares": 50, "price": 195.0},
    {"ticker": "MSFT", "shares": 30, "price": 410.0},
    {"ticker": "GOOGL", "shares": 10, "price": 175.0},
]

@app.get("/portfolio/export.csv")
def export_csv():
    lines = ["ticker,shares,price"]
    lines += [f"{h['ticker']},{h['shares']},{h['price']}" for h in PORTFOLIO]
    body = "\n".join(lines) + "\n"
    return Response(
        content=body,
        media_type="text/csv",
        headers={"Content-Disposition": 'attachment; filename="portfolio.csv"'},
    )

@app.delete("/assets/{ticker}", status_code=204)
def delete_asset(ticker: str):
    # 204 No Content — must not return a body. Returning Response() handles this cleanly.
    return Response(status_code=204)

client = TestClient(app)

r = client.get("/portfolio/export.csv")
print("status      :", r.status_code)
print("content-type:", r.headers["content-type"])
print("disposition :", r.headers["content-disposition"])
print("body:")
print(r.text)

r = client.delete("/assets/AAPL")
print("DELETE /assets/AAPL -> status:", r.status_code, "body:", repr(r.text))

## Key Takeaways

- **JSON in / JSON out is the default.** A `BaseModel`-typed argument is the body; the handler's return value gets JSON-serialized.
- **Path + query + body coexist** in one handler — FastAPI picks the slot by type.
- **Multiple body params** wrap into one JSON object keyed by parameter name. Use `Body(...)` for scalar singletons that need to ride in the body.
- **`response_model_exclude_unset=True`** is the right knob for PATCH responses — it omits untouched fields instead of returning defaults.
- **Aliases** (`Field(alias=...)` + `populate_by_name=True`) bridge a JSON name and a Python identifier without a mapping layer.
- **Drop to `Response`** for CSV, file downloads, 204 No Content, or anything else where JSON serialization is the wrong default.

## Exercises

**1. PATCH semantics.** Build `PATCH /assets/{ticker}` accepting an `AssetPatch` where every field is optional. The handler should:

- only update fields that were explicitly set in the request body (use `model_dump(exclude_unset=True)`).
- return only the changed fields, not the whole resource (use `response_model_exclude_unset=True`).

Send `{"price": 200}` and confirm the response is `{"price": 200}`, *not* a full `Asset`.

**2. CSV + JSON for the same resource.** Build two endpoints sharing the same underlying portfolio data:

- `GET /portfolio.json` — returns JSON (default).
- `GET /portfolio.csv` — returns CSV via `Response(..., media_type="text/csv")`.

Both should accept a `?limit=10` query param.

**3. Aliased payload.** Build `POST /clients` accepting a body with `firstName` and `lastName` (camelCase), backed by Python attributes `first_name` and `last_name`. Confirm via TestClient that the body submitted as JSON uses camelCase and the handler can still access `client.first_name`.